# Voice server on Kaggle (free, ~30 GPU hours per week)

1. In the right-hand panel → **Session options**: set **Accelerator** to **GPU T4 x2** and turn **Internet** on.
   (Kaggle asks you to verify your phone number once before GPU and Internet can be used.)
2. Pick the **MODEL** in the first cell (VoxCPM2 or IndexTTS-2.5), then **Run All**. The last cell prints a `https://….trycloudflare.com` URL — paste it into the app's **Settings → Server URL**.
3. Keep this tab open while you use the app. When you're done, click **Stop session** (power icon) so your weekly GPU hours aren't used while idle.

In [ ]:
# Which voice model should this server run?
MODEL = "VoxCPM2"  #@param ["VoxCPM2", "IndexTTS-2.5"]
# VoxCPM2:      default voice, voice design and cloning, 30 languages (~8 GB GPU memory).
# IndexTTS-2.5: voice cloning with emotions, English/Chinese/Japanese/Spanish/Arabic (~6 GB).
#               Needs a voice to clone (pick a file, record, or use My voices in the app).
# Optional: protect the server. Put the same key in the app's Settings.
API_KEY = ""  #@param {type:"string"}
assert MODEL in ("VoxCPM2", "IndexTTS-2.5"), MODEL

In [ ]:
!nvidia-smi -L
!wget -q -O server.py https://raw.githubusercontent.com/yassineallach/admin/claude/android-voxcpm-voice-app-8pkl1u/voxcpm-app/server/server.py
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared
if MODEL == "VoxCPM2":
    !pip -q install voxcpm fastapi "uvicorn[standard]" python-multipart soundfile librosa
else:
    # IndexTTS-2.5 needs its own Python 3.11 environment; uv builds it inside index-tts/.venv (a few minutes).
    !pip -q install -U uv
    ![ -d index-tts ] || git clone -q --depth 1 https://github.com/index-tts/index-tts.git
    !cd index-tts && uv sync -q
    !cd index-tts && uv pip install -q fastapi "uvicorn[standard]" python-multipart soundfile
    # ~5.5 GB of weights; helper models (~3 GB) download on the first start.
    !cd index-tts && .venv/bin/python -c "from huggingface_hub import snapshot_download; snapshot_download('IndexTeam/IndexTTS-2.5', local_dir='checkpoints')"

In [ ]:
import os, subprocess, time, urllib.request, urllib.error
env = dict(os.environ, VOXCPM_API_KEY=API_KEY)
if MODEL == "VoxCPM2":
    cmd, cwd = ['python', os.path.abspath('server.py'), '--port', '8000'], None
else:
    # Run from the index-tts folder: IndexTTS reads some files from relative paths.
    cmd = ['.venv/bin/python', os.path.abspath('server.py'), '--engine', 'indextts', '--model-dir', 'checkpoints', '--port', '8000']
    cwd = 'index-tts'
server = subprocess.Popen(cmd, cwd=cwd, env=env, stdout=open('server.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(900):
    if server.poll() is not None:
        break  # crashed, the log below says why
    try:
        urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=2); break
    except urllib.error.HTTPError:
        break  # answering (401 just means an API key is set)
    except Exception:
        time.sleep(2)
print(open('server.log').read()[-2000:])

In [ ]:
import re, subprocess, time
tunnel = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'], stdout=subprocess.DEVNULL, stderr=open('tunnel.log', 'w'))
for _ in range(60):
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('tunnel.log').read())
    if m:
        print('Server URL for the app:', m.group(0)); break
    time.sleep(1)
else:
    print(open('tunnel.log').read())